# 🇪🇹 Ethiopia-Only Malaria Early Warning System (MEWS)
### Climate-Driven EWS using real Ethiopian data sources — MSc Data Science, AAU

**Scope of this notebook:** every dataset pulled here is Ethiopia-specific — no generic Africa-wide
or global proxy data. This replaces the earlier version of the notebook, which used global/African
Kaggle datasets (Berkeley Earth global temperature, "Africa rainfall", pan-African malaria data) as
stand-ins. Those are removed.

**Real data sources used (all free, no paid API key required):**

| Domain | Source | Granularity | Access |
|---|---|---|---|
| Climate (temperature, rainfall, humidity) | **NASA POWER** (power.larc.nasa.gov) | Daily/monthly, per coordinate, 1981–present | Open REST API, no key |
| National malaria burden | **WHO Global Health Observatory (GHO)** OData API, `SpatialDim = ETH` | Annual, national | Open REST API, no key |
| ENSO / El Niño–La Niña phase | **NOAA PSL** Niño 3.4 index | Monthly | Open flat file |
| Regional 2023–2024 case/death share | WHO Disease Outbreak News (31 Oct 2024) + ReliefWeb, Oromia/Amhara/Southwest/South Ethiopia | Annual, regional | Cited, hand-entered (no public API for sub-national HMIS/PHEM data — see note in Cell 6) |

**Honest limitation, stated up front:** Ethiopia's real sub-national, monthly malaria case counts
live in EPHI's PHEM/HMIS system and are not exposed through a public API. This notebook builds the
EWS on (a) real Ethiopia-specific climate time series per region and (b) the real national trend +
the most recently published regional burden shares, and disaggregates the national trend by those
shares to approximate a regional monthly series. **For a thesis-grade result, replace `data/regional_malaria_monthly.csv`
in Cell 7 with actual EPHI PHEM/HMIS or DHS Program figures once you have institutional access** —
everything downstream (features, model, dashboard) is built to accept that swap without changes.


## 📦 Cell 1 — Install & Import

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — INSTALL AND IMPORT
# ════════════════════════════════════════════════════════════
!pip install -q requests pandas numpy scikit-learn matplotlib seaborn shap

import os, json, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

os.makedirs("data", exist_ok=True)
os.makedirs("data/dashboard", exist_ok=True)  # JSON exports for the GitHub-hosted dashboard

print("✅ Environment ready")


## 🗺️ Cell 2 — Ethiopian Regions Reference Table
Representative coordinates (regional capital / malaria-relevant town) for each of Ethiopia's current
regional states and two chartered city administrations, plus a literature-based malaria-ecology note.
These points are what we query NASA POWER against. For production use, swap single points for a
region-wide grid or zonal-mean raster (e.g. via Google Earth Engine, `ee.Image.reduceRegion`) —
noted in Cell 4.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — ETHIOPIA REGIONS (post-2023 administrative structure)
# ════════════════════════════════════════════════════════════
ETHIOPIA_REGIONS = {
    # region_name:      (lat,     lon,     representative_town,   ecological_zone,        national_2024_case_share_pct)
    "Oromia":                (8.5400, 39.2700, "Adama",          "mixed lowland/midland", 44.0),   # WHO DON 31-Oct-2024
    "Amhara":                (11.5936, 37.3908, "Bahir Dar",     "midland/highland fringe",18.0),   # WHO DON 31-Oct-2024
    "South West Ethiopia Peoples": (7.2667, 36.2333, "Bonga",    "humid lowland",          12.0),   # WHO DON "Southwest"
    "South Ethiopia":       (6.0333, 37.5500, "Arba Minch",      "rift valley lowland",     7.0),   # WHO DON 31-Oct-2024
    "Gambela":               (8.2500, 34.5833, "Gambela",        "humid lowland (hyperendemic)", 6.0),  # residual apportionment, see Cell 6
    "Benishangul-Gumuz":     (10.0667, 34.5333, "Assosa",        "humid lowland",           4.0),   # residual apportionment
    "Sidama":                (7.0500, 38.4667, "Hawassa",        "midland/highland fringe", 3.0),   # residual apportionment
    "SNNP":                  (6.8500, 37.7667, "Wolaita Sodo",   "midland",                 2.5),   # residual apportionment
    "Tigray":                (13.4967, 39.4753, "Mekelle",       "western lowland fringe",  2.0),   # residual apportionment
    "Somali":                (9.3500, 42.8000, "Jijiga",         "arid/semi-arid",          0.8),   # residual apportionment
    "Afar":                  (11.7952, 41.0117, "Semera",        "arid",                    0.4),   # residual apportionment
    "Harari":                (9.3132, 42.1181, "Harar",          "highland fringe",         0.2),   # residual apportionment
    "Dire Dawa":             (9.5931, 41.8661, "Dire Dawa",      "lowland/urban",           0.1),   # residual apportionment
    "Addis Ababa":           (9.0300, 38.7400, "Addis Ababa",    "highland (>2300m, minimal local transmission)", 0.0),
}

regions_df = pd.DataFrame([
    {"region": k, "lat": v[0], "lon": v[1], "town": v[2], "eco_zone": v[3], "case_share_pct_2024": v[4]}
    for k, v in ETHIOPIA_REGIONS.items()
])
regions_df.to_csv("data/ethiopia_regions_reference.csv", index=False)
print(f"✅ {len(regions_df)} Ethiopian regions/city-administrations defined")
regions_df


## 🌦️ Cell 3 — Fetch Real Ethiopian Climate Data (NASA POWER)
NASA POWER (`power.larc.nasa.gov`) is a public, key-free REST API returning satellite/reanalysis-
derived meteorology for any point on Earth since 1981. We query it once per Ethiopian region for:
`T2M` (mean temp °C), `T2M_MAX`, `T2M_MIN`, `PRECTOTCORR` (precipitation mm/day), `RH2M` (relative
humidity %) — the standard malaria-climate driver set (temperature governs parasite/vector
development rate, rainfall governs breeding-site availability, humidity governs vector survival).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — NASA POWER CLIMATE DATA (per Ethiopian region, monthly, 2010–present)
# ════════════════════════════════════════════════════════════
POWER_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
PARAMS = "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M"
START_YEAR, END_YEAR = 2010, date.today().year

def fetch_power_monthly(lat, lon, start_year=START_YEAR, end_year=END_YEAR):
    r = requests.get(POWER_URL, params={
        "parameters": PARAMS,
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start_year,
        "end": end_year,
        "format": "JSON",
    }, timeout=30)
    r.raise_for_status()
    payload = r.json()["properties"]["parameter"]
    rows = []
    for param, series in payload.items():
        for yyyymm, val in series.items():
            if yyyymm.endswith("13"):   # "13" = annual average row NASA POWER appends — skip
                continue
            rows.append({"yyyymm": yyyymm, "param": param, "value": val})
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["yyyymm"], format="%Y%m")
    wide = df.pivot_table(index="date", columns="param", values="value").reset_index()
    return wide

climate_frames = []
for region, (lat, lon, town, eco, share) in ETHIOPIA_REGIONS.items():
    print(f"⬇️  NASA POWER: {region} ({town}, {lat:.2f},{lon:.2f}) ...")
    try:
        wide = fetch_power_monthly(lat, lon)
        wide.insert(0, "region", region)
        climate_frames.append(wide)
        time.sleep(0.5)  # be polite to the API
    except Exception as e:
        print(f"   ⚠️  failed for {region}: {e}")

climate_df = pd.concat(climate_frames, ignore_index=True)
climate_df = climate_df.rename(columns={
    "T2M": "temp_mean_c", "T2M_MAX": "temp_max_c", "T2M_MIN": "temp_min_c",
    "PRECTOTCORR": "rainfall_mm_day", "RH2M": "humidity_pct",
})
climate_df["rainfall_mm_month"] = climate_df["rainfall_mm_day"] * climate_df["date"].dt.days_in_month
climate_df.to_csv("data/ethiopia_climate_monthly.csv", index=False)
print(f"✅ Climate panel: {len(climate_df)} region-months across {climate_df.region.nunique()} regions")
climate_df.head()


## 🩺 Cell 4 — Fetch Ethiopia's National Malaria Indicators (WHO GHO API)
The WHO Global Health Observatory OData API (`ghoapi.azureedge.net`) is public and key-free.
Rather than hard-coding an indicator code that might change, we first **discover** every indicator
whose name contains "Malaria", then pull each one filtered to `SpatialDim eq 'ETH'` (Ethiopia).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — WHO GHO API — ETHIOPIA MALARIA INDICATORS (national, annual)
# ════════════════════════════════════════════════════════════
GHO_BASE = "https://ghoapi.azureedge.net/api"

# Step 1: discover current malaria indicator codes
ind_resp = requests.get(f"{GHO_BASE}/Indicator", params={"$filter": "contains(IndicatorName,'Malaria')"}, timeout=30)
malaria_indicators = pd.DataFrame(ind_resp.json()["value"])
print(f"Found {len(malaria_indicators)} malaria-related indicators")
display_cols = ["IndicatorCode", "IndicatorName"]
print(malaria_indicators[display_cols].to_string(index=False))

# Step 2: pull Ethiopia (ISO3 = ETH) time series for each
who_frames = []
for _, row in malaria_indicators.iterrows():
    code_, name = row["IndicatorCode"], row["IndicatorName"]
    try:
        r = requests.get(f"{GHO_BASE}/{code_}", params={"$filter": "SpatialDim eq \'ETH\'"}, timeout=30)
        vals = r.json().get("value", [])
        if not vals:
            continue
        df = pd.DataFrame(vals)[["TimeDim", "NumericValue", "Dim1"]].copy()
        df["IndicatorCode"] = code_
        df["IndicatorName"] = name
        who_frames.append(df)
    except Exception as e:
        print(f"   ⚠️  {code_} failed: {e}")

who_ethiopia = pd.concat(who_frames, ignore_index=True) if who_frames else pd.DataFrame()
who_ethiopia = who_ethiopia.rename(columns={"TimeDim": "year", "NumericValue": "value"})
who_ethiopia.to_csv("data/ethiopia_who_malaria_national.csv", index=False)
print(f"✅ WHO GHO Ethiopia malaria records: {len(who_ethiopia)}")
who_ethiopia.sort_values(["IndicatorName", "year"]).head(20)


## 🌊 Cell 5 — ENSO Niño 3.4 Index (NOAA)
East African "short rains" (Belg) and the broader Horn of Africa rainfall regime are modulated by
ENSO. Included as a regional/teleconnection covariate, not a substitute for Ethiopia-local data.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — NOAA ENSO NIÑO 3.4 INDEX
# ════════════════════════════════════════════════════════════
def fetch_enso_noaa():
    url = "https://psl.noaa.gov/data/correlation/nina34.data"
    resp = requests.get(url, timeout=20)
    records = []
    for line in resp.text.strip().split("\n"):
        parts = line.split()
        if len(parts) == 13:
            try:
                year = int(parts[0])
                for m, v in enumerate(parts[1:], start=1):
                    val = float(v)
                    if abs(val) < 90:
                        records.append({"year": year, "month": m, "enso_nino34": val})
            except ValueError:
                pass
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df[["year", "month"]].assign(day=1))
    return df.sort_values("date").reset_index(drop=True)

enso_df = fetch_enso_noaa()
enso_df.to_csv("data/enso_nino34.csv", index=False)
print(f"✅ ENSO series: {len(enso_df)} months ({enso_df.year.min()}–{enso_df.year.max()})")
enso_df.tail()


## 📋 Cell 6 — Regional Burden Apportionment (documented, cited)
Ethiopia's public national trend (Cell 4) is annual/national only. To approximate a **regional**
monthly series without access to EPHI PHEM/HMIS microdata, we apportion the national monthly trend
by each region's most recently published case-share:

- **Oromia 44%, Amhara 18%, Southwest 12%, South Ethiopia 7%** — directly from the WHO Disease
  Outbreak News on Ethiopia malaria, 31 October 2024 (covering Jan 1–Oct 20, 2024; these four
  regions = 81% of national cases and 89% of malaria deaths that year).
- The remaining ~19% is spread across Gambela, Benishangul-Gumuz, Sidama, SNNP, Tigray, Somali,
  Afar, Harari, Dire Dawa and Addis Ababa using published endemicity literature (Gambela and
  Benishangul-Gumuz are hyperendemic humid lowlands despite low population share; Addis Ababa,
  >2,300m, has effectively no local transmission). **These residual shares are estimated, not an
  official statistic** — replace with EPHI/PHEM regional figures when available.

This is a stated modelling assumption, not ground truth — flagged again in the EWS output.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — BUILD APPROXIMATE REGIONAL MONTHLY MALARIA SERIES
# ════════════════════════════════════════════════════════════
# National monthly proxy: WHO GHO gives annual estimates: interpolate to monthly using the
# historically documented Ethiopia malaria seasonality (peak Sep-Dec after kiremt rains, a smaller
# peak Mar-May after belg rains) rather than a flat/uniform spread.
SEASONAL_WEIGHTS = {  # relative monthly share of an "average" transmission year, sums to 1.0
    1: 0.055, 2: 0.05, 3: 0.06, 4: 0.07, 5: 0.075, 6: 0.06,
    7: 0.055, 8: 0.06, 9: 0.11, 10: 0.15, 11: 0.135, 12: 0.10,
}
assert abs(sum(SEASONAL_WEIGHTS.values()) - 1.0) < 1e-6

national_cases = who_ethiopia[who_ethiopia["IndicatorName"].str.contains("cases", case=False, na=False)]
national_cases = national_cases.dropna(subset=["value"]).groupby("year", as_index=False)["value"].max()
national_cases = national_cases.rename(columns={"value": "national_est_cases"})

monthly_national = []
for _, row in national_cases.iterrows():
    for m, w in SEASONAL_WEIGHTS.items():
        monthly_national.append({
            "date": pd.Timestamp(int(row["year"]), m, 1),
            "national_est_cases_month": row["national_est_cases"] * w,
        })
monthly_national_df = pd.DataFrame(monthly_national)

regional_monthly = []
for region, (lat, lon, town, eco, share_pct) in ETHIOPIA_REGIONS.items():
    tmp = monthly_national_df.copy()
    tmp["region"] = region
    tmp["case_share_pct"] = share_pct
    tmp["est_cases"] = tmp["national_est_cases_month"] * (share_pct / 100.0)
    regional_monthly.append(tmp)

regional_malaria_df = pd.concat(regional_monthly, ignore_index=True)
regional_malaria_df.to_csv("data/regional_malaria_monthly_APPROXIMATED.csv", index=False)
print("✅ Approximated regional-monthly malaria series built")
print("   ⚠️  Replace data/regional_malaria_monthly_APPROXIMATED.csv with real EPHI PHEM/HMIS")
print("      or DHS Program figures for thesis-grade validation.")
regional_malaria_df.head()


## 🔗 Cell 7 — Merge Into a Single Ethiopia Region-Month Panel

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — MERGE CLIMATE + MALARIA + ENSO INTO ONE PANEL
# ════════════════════════════════════════════════════════════
panel = climate_df.merge(regional_malaria_df[["date", "region", "est_cases", "case_share_pct"]],
                          on=["date", "region"], how="left")
panel = panel.merge(enso_df[["date", "enso_nino34"]], on="date", how="left")
panel = panel.sort_values(["region", "date"]).reset_index(drop=True)
panel.to_csv("data/ethiopia_malaria_climate_panel.csv", index=False)
print(f"✅ Final panel: {panel.shape[0]} rows × {panel.shape[1]} cols, "
      f"{panel.region.nunique()} regions, {panel.date.min().date()}–{panel.date.max().date()}")
panel.head()


## 📊 Cell 8 — Exploratory Data Analysis (Ethiopia-only)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — EDA
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# National malaria trend (real WHO figures)
nat_yearly = national_cases.copy()
axes[0,0].bar(nat_yearly["year"], nat_yearly["national_est_cases"]/1e6, color="#c0392b")
axes[0,0].set_title("Ethiopia — WHO-reported malaria cases per year (millions)")
axes[0,0].set_ylabel("Million cases")

# Regional case-share
rs = regions_df.sort_values("case_share_pct_2024", ascending=True)
axes[0,1].barh(rs["region"], rs["case_share_pct_2024"], color="#1a7a4a")
axes[0,1].set_title("Regional case-share, 2024 (WHO DON, cited; residual = estimated)")
axes[0,1].set_xlabel("% of national cases")

# Rainfall seasonality by region (mean by calendar month)
climate_df["month"] = climate_df["date"].dt.month
seas = climate_df.groupby(["region", "month"])["rainfall_mm_month"].mean().reset_index()
for region in seas.region.unique()[:6]:
    sub = seas[seas.region == region]
    axes[1,0].plot(sub["month"], sub["rainfall_mm_month"], marker="o", label=region)
axes[1,0].set_title("Monthly rainfall seasonality (subset of regions)")
axes[1,0].set_xlabel("Month"); axes[1,0].legend(fontsize=8)

# Temperature vs ENSO overlay for one region
sample = panel[panel.region == "Oromia"]
ax2 = axes[1,1].twinx()
axes[1,1].plot(sample["date"], sample["rainfall_mm_month"], color="#0891b2", label="Rainfall (mm/mo)")
ax2.plot(sample["date"], sample["enso_nino34"], color="#d4860a", label="ENSO Niño3.4")
axes[1,1].set_title("Oromia — rainfall vs ENSO phase")
axes[1,1].legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("data/dashboard/eda_overview.png", dpi=120)
plt.show()


## 🧪 Cell 9 — Feature Engineering
Climate→malaria transmission has a known biological lag (mosquito development + parasite
incubation + reporting delay), so we build 1–3 month lagged rainfall/temperature/humidity features,
a Kiremt/Belg/Bega seasonal flag, and a rainfall anomaly (z-score vs that region's own climatology).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════
def season_of(month):
    if month in (6, 7, 8, 9):
        return "kiremt"       # main rainy season
    if month in (2, 3, 4, 5):
        return "belg"         # short rains
    return "bega"             # dry season

feat = panel.copy()
feat["month"] = feat["date"].dt.month
feat["season"] = feat["month"].apply(season_of)

feat = feat.sort_values(["region", "date"])
for lag in (1, 2, 3):
    feat[f"rainfall_lag{lag}"] = feat.groupby("region")["rainfall_mm_month"].shift(lag)
    feat[f"temp_lag{lag}"] = feat.groupby("region")["temp_mean_c"].shift(lag)
    feat[f"humidity_lag{lag}"] = feat.groupby("region")["humidity_pct"].shift(lag)

clim = feat.groupby("region")["rainfall_mm_month"].transform("mean")
clim_std = feat.groupby("region")["rainfall_mm_month"].transform("std")
feat["rainfall_anomaly_z"] = (feat["rainfall_mm_month"] - clim) / clim_std

feat = pd.get_dummies(feat, columns=["season"], prefix="season")
feat = feat.dropna(subset=["rainfall_lag3", "temp_lag3", "humidity_lag3"]).reset_index(drop=True)
feat.to_csv("data/ethiopia_malaria_features.csv", index=False)
print(f"✅ Feature table: {feat.shape[0]} rows × {feat.shape[1]} cols")
feat.head()


## 🚨 Cell 10 — EWS Risk Scoring (rule-based) + ML Classifier
Two layers, as is standard for operational EWS design:

1. **Rule-based risk score** (transparent, auditable, usable even with no ML): combines rainfall
   anomaly, lagged temperature suitability (18–32°C optimal for *P. falciparum/vivax* transmission),
   ENSO phase and the region's own endemicity weight.
2. **ML classifier (Random Forest)**, trained to predict whether a region-month's estimated case
   count exceeds its own historical 75th percentile ("elevated risk"), using only lagged climate
   features (no same-month case data — this keeps it a genuine *early warning*, not nowcasting).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — RISK SCORE + ML CLASSIFIER
# ════════════════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# --- Rule-based score (0-100) ---
def temp_suitability(t):
    # simple triangular suitability curve, optimum ~25C
    if t < 16 or t > 34:
        return 0.0
    return max(0.0, 1 - abs(t - 25) / 9)

feat["temp_suitability"] = feat["temp_lag1"].apply(temp_suitability)
feat["rain_score"] = feat["rainfall_anomaly_z"].clip(-2, 2).add(2).div(4)     # 0-1
feat["enso_score"] = feat["enso_nino34"].clip(-2, 2).add(2).div(4)           # 0-1 (La Niña often wetter Horn of Africa)
feat["endemicity_score"] = feat["case_share_pct"].fillna(0) / feat["case_share_pct"].max()

feat["risk_score"] = (
    0.35 * feat["rain_score"] +
    0.30 * feat["temp_suitability"] +
    0.15 * feat["enso_score"] +
    0.20 * feat["endemicity_score"]
) * 100

def risk_level(s):
    if s >= 70: return "Alert"
    if s >= 50: return "Warning"
    if s >= 30: return "Watch"
    return "Low"

feat["risk_level"] = feat["risk_score"].apply(risk_level)

# --- ML classifier: predict "elevated" months from lagged climate only ---
feat["case_p75_region"] = feat.groupby("region")["est_cases"].transform(lambda s: s.quantile(0.75))
feat["elevated_label"] = (feat["est_cases"] > feat["case_p75_region"]).astype(int)

X_cols = [c for c in feat.columns if c.startswith(("rainfall_lag", "temp_lag", "humidity_lag", "season_"))] + ["enso_nino34"]
X = feat[X_cols].fillna(0)
y = feat["elevated_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
try:
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
except Exception:
    pass

feat.to_csv("data/ethiopia_malaria_features.csv", index=False)
feat[["region", "date", "risk_score", "risk_level"]].tail(15)


## 📈 Cell 11 — Model Evaluation & Feature Importance

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — FEATURE IMPORTANCE
# ════════════════════════════════════════════════════════════
importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=True)
plt.figure(figsize=(9, 6))
importances.plot(kind="barh", color="#0d1b3e")
plt.title("Feature importance — elevated-risk classifier (Ethiopia, lagged climate only)")
plt.tight_layout()
plt.savefig("data/dashboard/feature_importance.png", dpi=120)
plt.show()


## 📤 Cell 12 — EWS Alert Bulletin + Export for the GitHub Dashboard
Writes the JSON files the HTML dashboard (`Ethiopia_MEWS_Dashboard.html`) reads directly:
`data/dashboard/national_trend.json`, `regional_risk.json`, `climate_latest.json`, `meta.json`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — CURRENT ALERT BULLETIN + DASHBOARD JSON EXPORT
# ════════════════════════════════════════════════════════════
latest_date = feat["date"].max()
latest = feat[feat["date"] == latest_date].sort_values("risk_score", ascending=False)

print(f"🚨 ETHIOPIA MALARIA EWS BULLETIN — {latest_date.strftime('%B %Y')}\n")
for _, r in latest.iterrows():
    print(f"  {r['region']:<32} risk={r['risk_score']:5.1f}  level={r['risk_level']:<8} "
          f"(rain_anom_z={r['rainfall_anomaly_z']:+.2f}, temp_lag1={r['temp_lag1']:.1f}°C)")

# --- Dashboard JSON: national trend ---
national_trend_json = [
    {"year": int(r.year), "cases": float(r.national_est_cases)}
    for r in national_cases.itertuples()
]
with open("data/dashboard/national_trend.json", "w") as f:
    json.dump(national_trend_json, f, indent=2)

# --- Dashboard JSON: regional risk (latest month) ---
regional_risk_json = [
    {
        "region": r["region"], "town": ETHIOPIA_REGIONS[r["region"]][2],
        "lat": ETHIOPIA_REGIONS[r["region"]][0], "lon": ETHIOPIA_REGIONS[r["region"]][1],
        "risk_score": round(float(r["risk_score"]), 1), "risk_level": r["risk_level"],
        "rainfall_anomaly_z": round(float(r["rainfall_anomaly_z"]), 2),
        "temp_c": round(float(r["temp_lag1"]), 1),
        "case_share_pct": float(r["case_share_pct"]) if pd.notna(r["case_share_pct"]) else None,
    }
    for _, r in latest.iterrows()
]
with open("data/dashboard/regional_risk.json", "w") as f:
    json.dump(regional_risk_json, f, indent=2)

# --- Dashboard JSON: climate time series (last 36 months, all regions) ---
recent = feat[feat["date"] >= latest_date - pd.DateOffset(months=36)]
climate_latest_json = [
    {"region": r["region"], "date": r["date"].strftime("%Y-%m-%d"),
     "rainfall_mm": round(float(r["rainfall_mm_month"]), 1),
     "temp_c": round(float(r["temp_mean_c"]), 1),
     "enso": round(float(r["enso_nino34"]), 2) if pd.notna(r["enso_nino34"]) else None}
    for _, r in recent.iterrows()
]
with open("data/dashboard/climate_latest.json", "w") as f:
    json.dump(climate_latest_json, f, indent=2)

# --- Meta ---
meta_json = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "latest_data_month": latest_date.strftime("%Y-%m"),
    "sources": ["NASA POWER (climate)", "WHO GHO OData API (national malaria)",
                "NOAA Niño 3.4 (ENSO)", "WHO Disease Outbreak News 31-Oct-2024 (regional shares)"],
    "note": "Regional-monthly malaria case series is an apportionment of the national WHO trend by "
            "published regional case-share, not raw EPHI PHEM/HMIS data. Replace for thesis-grade validation.",
}
with open("data/dashboard/meta.json", "w") as f:
    json.dump(meta_json, f, indent=2)

print("\n✅ Dashboard JSON exported to data/dashboard/ — ready to commit to GitHub (see Cell 13)")


## 🔁 Cell 13 — Push `data/dashboard/*.json` to Your GitHub Repo
Run this in Colab to publish fresh data to a GitHub Pages–hosted dashboard. Requires a
[fine-grained personal access token](https://github.com/settings/tokens) with `contents: write`
on the target repo. **Never paste a real token into a notebook you'll share** — use
`getpass` (below) so it isn't saved in the `.ipynb` file's output.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — PUBLISH TO GITHUB (run manually, not part of automated re-runs)
# ════════════════════════════════════════════════════════════
from getpass import getpass

GITHUB_USER = "YOUR_GITHUB_USERNAME"          # <-- edit
GITHUB_REPO = "ethiopia-malaria-ews"          # <-- edit: your repo name
GITHUB_BRANCH = "main"

token = getpass("GitHub personal access token (input hidden): ")

!git config --global user.email "colab@example.com"
!git config --global user.name "Ethiopia MEWS Colab"
!rm -rf repo_clone
!git clone -q https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git repo_clone
!mkdir -p repo_clone/data/dashboard
!cp data/dashboard/*.json repo_clone/data/dashboard/
!cd repo_clone && git add data/dashboard/*.json && \
  git commit -m "Update Ethiopia MEWS data - $(date -u +%Y-%m-%dT%H:%M:%SZ)" && \
  git push origin {GITHUB_BRANCH}

print("✅ Pushed — if GitHub Pages is enabled on this repo, the dashboard updates within a minute or two.")
